In [ ]:
!pip install open3d datasets plyfile huggingface_hub plotly wandb

In [ ]:
!pip install torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.0 MB/s eta 0:00:00


DOWNLOADING

In [ ]:

import os
from google.colab import userdata
os.environ['WANDB_API_KEY']=userdata.get('WANDB_API_KEY')
os.environ['HF_TOKEN']=userdata.get('HUGGINGFACE')


from huggingface_hub import hf_hub_download, snapshot_download

# Downloading the dataset
hf_hub_download(
    repo_id="BGLab/AgriField3D",
    filename="datasets/FielGrwon_ZeaMays_RawPCD_10k.zip",
    repo_type="dataset",
    local_dir="/content/MaizeField3D"
)

datasets/FielGrwon_ZeaMays_RawPCD_10k.zi(…):   0%|          | 0.00/218M [00:00<?, ?B/s]

'/content/MaizeField3D/datasets/FielGrwon_ZeaMays_RawPCD_10k.zip'

UNZIPPING

In [ ]:
import zipfile, os

zip_path = "/content/MaizeField3D/datasets/FielGrwon_ZeaMays_RawPCD_10k.zip"
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("./MaizeField3D/extracted")

print("Unzipped!")

Unzipped!


In [ ]:
import os

data_path = "/content/MaizeField3D/extracted"

ply_files = []

for root, dirs, files in os.walk(data_path):
    for file in files:
        if file.endswith(".ply"):
            ply_files.append(os.path.join(root, file))

print("Total PLY files:", len(ply_files))

Total PLY files: 1045


#DGCN autoencoder for point cloud

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import open3d as o3d
import numpy as np
import wandb

# DATASET
class PointCloudDataset(Dataset):

    def __init__(self, ply_files, num_points=2048):
        self.files = ply_files
        self.num_points = num_points

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):

        pcd = o3d.io.read_point_cloud(self.files[idx])
        points = np.asarray(pcd.points)

        if points.shape[0] > self.num_points:
            choice = np.random.choice(points.shape[0], self.num_points, replace=False)
            points = points[choice]

        center = points.mean(axis=0)
        points = points - center
        scale = np.max(np.linalg.norm(points, axis=1))
        points = points / scale

        return torch.tensor(points, dtype=torch.float32)

def knn(x, k):

    inner = -2 * torch.matmul(x.transpose(2,1), x)
    xx = torch.sum(x**2, dim=1, keepdim=True)
    pairwise_distance = -xx - inner - xx.transpose(2,1)

    idx = pairwise_distance.topk(k=k, dim=-1)[1]

    return idx

#  EDGE FEATURE
def get_graph_feature(x, k=20):

    B, C, N = x.shape
    idx = knn(x, k)

    device = x.device
    idx_base = torch.arange(0, B, device=device).view(-1,1,1)*N
    idx = idx + idx_base
    idx = idx.view(-1)

    x = x.transpose(2,1).contiguous()
    feature = x.view(B*N, -1)[idx,:]
    feature = feature.view(B, N, k, C)
    x = x.view(B, N, 1, C).repeat(1,1,k,1)

    feature = torch.cat((feature-x, x), dim=3).permute(0,3,1,2)

    return feature

model

In [ ]:
#  DGCNN AUTOENCODER
class DGCNNAutoEncoder(nn.Module):

    def __init__(self, num_points=2048, latent_dim=128):
        super().__init__()

        self.k = 20
        self.num_points = num_points

        self.conv1 = nn.Sequential(
            nn.Conv2d(6,64,1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2)
        )

        self.conv2 = nn.Sequential(
            nn.Conv2d(128,64,1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2)
        )

        self.conv3 = nn.Sequential(
            nn.Conv2d(128,128,1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2)
        )

        self.conv4 = nn.Sequential(
            nn.Conv2d(256,256,1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2)
        )

        self.fc1 = nn.Linear(512, latent_dim)

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim,512),
            nn.ReLU(),
            nn.Linear(512,1024),
            nn.ReLU(),
            nn.Linear(1024,num_points*3)
        )

    def forward(self,x):

        B,N,_ = x.shape
        x = x.permute(0,2,1)

        x1 = get_graph_feature(x, self.k)
        x1 = self.conv1(x1)
        x1 = x1.max(dim=-1)[0]

        x2 = get_graph_feature(x1, self.k)
        x2 = self.conv2(x2)
        x2 = x2.max(dim=-1)[0]

        x3 = get_graph_feature(x2, self.k)
        x3 = self.conv3(x3)
        x3 = x3.max(dim=-1)[0]

        x4 = get_graph_feature(x3, self.k)
        x4 = self.conv4(x4)
        x4 = x4.max(dim=-1)[0]

        x_cat = torch.cat((x1,x2,x3,x4), dim=1)
        x_global = F.adaptive_max_pool1d(x_cat,1).view(B,-1)

        z = self.fc1(x_global)

        out = self.decoder(z)
        out = out.view(B,self.num_points,3)

        return out

In [ ]:
#  CHAMFER LOSS
def chamfer_distance(p1, p2):

    dist = torch.cdist(p1, p2)

    loss1 = dist.min(dim=2)[0].mean()
    loss2 = dist.min(dim=1)[0].mean()

    return loss1 + loss2

In [ ]:
ply_files=ply_files[:500]

dataset = PointCloudDataset(ply_files)

train_set, test_set = torch.utils.data.random_split(dataset,[400,100])

train_loader = DataLoader(train_set,batch_size=8,shuffle=True)
test_loader = DataLoader(test_set,batch_size=8)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = DGCNNAutoEncoder().to(device)
optimizer = torch.optim.Adam(model.parameters(),lr=1e-3)

In [ ]:
wandb.init(project="ppt_mesh_autoencoder")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: captainyagami08 (captainyagami08-tybi) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


training and testing

In [ ]:
for epoch in range(50):

    model.train()
    train_loss = 0

    for points in train_loader:

        points = points.to(device)

        recon = model(points)

        loss = chamfer_distance(points, recon)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    model.eval()
    test_loss = 0

    with torch.no_grad():

        for points in test_loader:

            points = points.to(device)

            recon = model(points)

            loss = chamfer_distance(points, recon)

            test_loss += loss.item()

    test_loss /= len(test_loader)

    wandb.log({
        "epoch":epoch,
        "train_loss":train_loss,
        "test_loss":test_loss
    })

    print(f"Epoch:{epoch} Train:{train_loss} Test:{test_loss}")

Epoch:0 Train:0.12666006579995157 Test:0.11459840604892144
Epoch:1 Train:0.10577813312411308 Test:0.10381790823661365
Epoch:2 Train:0.1015209673345089 Test:0.09621991102512066
Epoch:3 Train:0.09870750471949577 Test:0.09513666996589074
Epoch:4 Train:0.09801499813795089 Test:0.09653368477637951
Epoch:5 Train:0.0957041797041893 Test:0.09273020808513348
Epoch:6 Train:0.09238667786121368 Test:0.09168255615692872
Epoch:7 Train:0.09219263538718224 Test:0.09601666377140926
Epoch:8 Train:0.09261346235871315 Test:0.09123301907227589
Epoch:9 Train:0.08989103585481643 Test:0.0943065331532405
Epoch:10 Train:0.08897420972585678 Test:0.08883699201620542
Epoch:11 Train:0.08801503613591194 Test:0.09226691207060447
Epoch:12 Train:0.08792153343558312 Test:0.0882435767696454
Epoch:13 Train:0.08723145455121994 Test:0.09081363792602833
Epoch:14 Train:0.08771025002002716 Test:0.09191364565720925
Epoch:15 Train:0.0878677423298359 Test:0.09160434913176757
Epoch:16 Train:0.08571990221738815 Test:0.0871010663417

In [ ]:
import random
import open3d as o3d
import numpy as np
import torch
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def visualize_random_reconstruction(ply_files, model, device, num_points=2048):

    model.eval()

    file = random.choice(ply_files)

    pcd = o3d.io.read_point_cloud(file)

    points = np.asarray(pcd.points)
    colors = np.asarray(pcd.colors)

    idx = np.random.choice(points.shape[0], num_points, replace=False)
    points = points[idx]
    colors = colors[idx]

    center = points.mean(axis=0)
    points = points - center
    scale = np.max(np.linalg.norm(points, axis=1))
    points = points / scale

    points_tensor = torch.tensor(points, dtype=torch.float32).unsqueeze(0).to(device)

    with torch.no_grad():
        recon = model(points_tensor)

    recon = recon.squeeze(0).cpu().numpy()

    fig = make_subplots(rows=1, cols=2,
                        specs=[[{"type":"scene"},{"type":"scene"}]],
                        subplot_titles=("Original","Reconstructed"))

    fig.add_trace(
        go.Scatter3d(
            x=points[:,0],
            y=points[:,1],
            z=points[:,2],
            mode="markers",
            marker=dict(size=2, color=(colors*255).astype(int))
        ),
        row=1, col=1
    )

    fig.add_trace(
        go.Scatter3d(
            x=recon[:,0],
            y=recon[:,1],
            z=recon[:,2],
            mode="markers",
            marker=dict(size=2, color=(colors*255).astype(int))
        ),
        row=1, col=2
    )

    fig.show()


visualize_random_reconstruction(ply_files, model, device)